In [ ]:
!pip install -q "transformers>=4.55" datasets accelerate scikit-learn matplotlib

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)

assert torch.cuda.is_available(), "Увімкніть T4 GPU у налаштуваннях Colab."

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Пам'ять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Задача і чому потрібне донавчання

**Задача.** Автоматично класифікувати короткі англомовні повідомлення за шістьма емоціями: `sadness`, `joy`, `love`, `anger`, `fear`, `surprise`.

**Практичне застосування.** Система аналізу зворотного зв’язку може передавати повідомлення з емоціями `anger` або `fear` оператору для швидшої реакції, а інші повідомлення — до аналітики якості сервісу. На вхід подається текст, на виході — одна з шести міток.

**Чому потрібне донавчання.** Промпт для генеративної моделі не гарантує одну мітку з фіксованого переліку, а довгу інструкцію довелося б повторювати в кожному запиті. RAG також не розв’язує задачу, адже моделі не бракує нових фактів — потрібна стабільна поведінка класифікатора. Тому використано донавчання на розмічених прикладах «текст → емоція».


## Модель і дані

**Модель:** `distilbert-base-uncased` — компактна англомовна encoder-модель приблизно на 67 млн параметрів. Вона придатна для класифікації коротких текстів, швидко навчається й уміщується в безкоштовну GPU T4.

**Дані:** [`dair-ai/emotion`](https://huggingface.co/datasets/dair-ai/emotion), ліцензія CC BY 4.0. Формат: `text` і `label`. Використовується готовий авторський поділ: 16 000 прикладів для training, 2 000 для validation і 2 000 для test. Test-набір не використовується для навчання або вибору гіперпараметрів.


In [ ]:
# Завантажуємо датасет і назви шести класів.
dataset = load_dataset("dair-ai/emotion", "split")
label_names = dataset["train"].features["label"].names

train_dataset = dataset["train"]
validation_dataset = dataset["validation"]
test_dataset = dataset["test"]

print(dataset)
print(f"Класи: {label_names}")
print()

# Показуємо п'ять прикладів із training-набору.
for example in train_dataset.select(range(5)):
    emotion = label_names[example["label"]]
    print(f"{emotion:>8}: {example['text']}")


## Метрика й результат до донавчання

Основна метрика — **macro-F1**. Вона обчислює F1 окремо для кожної емоції, а потім усереднює значення, тому всі шість класів мають однакову вагу. Додатково використовується **accuracy** — частка правильних прогнозів.

До донавчання використано baseline: завжди передбачати найчастішу емоцію у training-наборі. Це чесна найпростіша точка відліку для класифікатора з новою головою.


In [ ]:
# Обчислюємо baseline "завжди найчастіший клас" на test-наборі.
train_labels = np.array(train_dataset["label"])
test_labels = np.array(test_dataset["label"])

majority_label = int(np.bincount(train_labels).argmax())
baseline_predictions = np.full_like(test_labels, majority_label)

before_metrics = {
    "Модель": "Baseline: найчастіший клас",
    "Мітка baseline": label_names[majority_label],
    "Accuracy": accuracy_score(test_labels, baseline_predictions),
    "Macro-F1": f1_score(test_labels, baseline_predictions, average="macro"),
}

display(pd.DataFrame([before_metrics]))


## Метод донавчання й гіперпараметри

Застосовано **класичне повне донавчання з новою головою**: `AutoModelForSequenceClassification` додає до DistilBERT лінійний шар на шість виходів. Під час навчання оновлюються і backbone, і ця голова.

Гіперпараметри: `learning_rate=2e-5`, `num_train_epochs=3`, training batch size 32, evaluation batch size 64, `weight_decay=0.01`, змішана точність `fp16=True`. Малий learning rate потрібний, щоб не зруйнувати знання, отримані під час попереднього тренування.


In [ ]:
# Токенізуємо тексти. Довгі повідомлення обрізаємо до 128 токенів.
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
# Створюємо модель із новою головою на шість емоцій.
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(label_names),
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
    }


In [ ]:
# Налаштовуємо Trainer.
training_args = TrainingArguments(
    output_dir="emotion-ft",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=True,
    logging_steps=50,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [ ]:
# Вимірюємо час і пікове використання GPU під час донавчання.
torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()
trainer.train()
training_minutes = (time.perf_counter() - start_time) / 60
peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9

print(f"Тривалість навчання: {training_minutes:.2f} хв")
print(f"Пікова пам'ять GPU: {peak_memory_gb:.2f} GB")


In [ ]:
# Будуємо графік training loss та validation loss.
history = pd.DataFrame(trainer.state.log_history)

train_history = history.dropna(subset=["loss", "epoch"])
eval_history = history.dropna(subset=["eval_loss", "epoch"])

display(history)

plt.figure(figsize=(8, 4))
plt.plot(train_history["epoch"], train_history["loss"], "o-", label="Training loss")
plt.plot(eval_history["epoch"], eval_history["eval_loss"], "o-", label="Validation loss")
plt.xlabel("Епоха")
plt.ylabel("Loss")
plt.title("Зміна loss під час донавчання")
plt.grid()
plt.legend()
plt.show()


## Результат після донавчання

Після завершення навчання модель перевіряється на **тому самому test-наборі**, що й baseline. У таблиці нижче порівнюються accuracy та macro-F1 до і після донавчання. Також виводяться десять реальних прогнозів моделі на test-прикладах.


In [ ]:
# Оцінюємо донавчену модель на test-наборі.
test_output = trainer.predict(tokenized_dataset["test"])
test_predictions = np.argmax(test_output.predictions, axis=-1)

after_metrics = {
    "Модель": "Fine-tuned DistilBERT",
    "Мітка baseline": "—",
    "Accuracy": accuracy_score(test_labels, test_predictions),
    "Macro-F1": f1_score(test_labels, test_predictions, average="macro"),
}

comparison = pd.DataFrame([before_metrics, after_metrics])
display(comparison)

print(f"Тривалість навчання: {training_minutes:.2f} хв")
print(f"Пікова пам'ять GPU: {peak_memory_gb:.2f} GB")


In [ ]:
# Показуємо десять прогнозів на test-наборі.
examples = test_dataset.select(range(10))

for text, actual, predicted in zip(
    examples["text"],
    examples["label"],
    test_predictions[:10],
):
    print(f"Текст: {text}")
    print(f"Правильна емоція: {label_names[actual]}")
    print(f"Прогноз моделі:   {label_names[predicted]}")
    print("-" * 70)


## Аналіз помилок

Для аналізу показано classification report, confusion matrix і десять помилкових прогнозів. Найскладнішими очікувано є близькі емоції, наприклад `joy` та `love`, або `anger` та `sadness`, а також короткі тексти без достатнього контексту.


In [ ]:
# Детальні метрики за кожним класом і матриця помилок.
print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=label_names,
        digits=3,
    )
)

ConfusionMatrixDisplay.from_predictions(
    test_labels,
    test_predictions,
    display_labels=label_names,
    xticks_rotation=45,
)
plt.title("Confusion matrix на test-наборі")
plt.show()


In [ ]:
# Виводимо до десяти прикладів, у яких модель помилилася.
error_indices = np.flatnonzero(test_predictions != test_labels)[:10]

print(f"Кількість помилок у test-наборі: {(test_predictions != test_labels).sum()}")

for index in error_indices:
    example = test_dataset[int(index)]
    print(f"Текст: {example['text']}")
    print(f"Правильна емоція: {label_names[example['label']]}")
    print(f"Прогноз моделі:   {label_names[test_predictions[index]]}")
    print("-" * 70)


## Висновок

Було донавчено DistilBERT для класифікації емоцій у коротких текстах. Результати у таблиці порівняння показують, наскільки донавчена модель перевищує baseline за accuracy та macro-F1. Графік loss і validation-метрики показують, чи навчання було стабільним і чи з’явилися ознаки перенавчання.

Після донавчання модель може отримувати лише текст і повертати одну з шести відомих міток, без довгого промпту з описом усіх класів. Ваги моделі до Git-репозиторію не додаються.


## Перевірка на власних прикладах

Перевіримо донавчену модель на нових реченнях, яких немає в test-наборі. Для кожного тексту виводиться прогнозована емоція та її імовірність.


In [ ]:
# Власні приклади для перевірки моделі.
my_examples = [
    "I am so happy that I passed my final exam!",
    "I feel nervous about tomorrow's job interview.",
    "This is terrible service and I am extremely angry.",
    "I miss my friends and feel lonely today.",
    "I did not expect such a wonderful birthday present!",
    "I really care about you and appreciate everything you do.",
]


In [ ]:
# Функція повертає найімовірнішу емоцію та її імовірність.
def predict_emotion(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to(model.device)

    with torch.no_grad():
        probabilities = model(**inputs).logits.softmax(dim=-1)[0]

    label_id = int(probabilities.argmax())
    return label_names[label_id], float(probabilities[label_id])


In [ ]:
# Виводимо результати для власних прикладів.
for text in my_examples:
    emotion, probability = predict_emotion(text)

    print(f"Текст: {text}")
    print(f"Прогноз: {emotion}")
    print(f"Імовірність: {probability:.1%}")
    print("-" * 70)
